# Web Scraping with Scrapy

**Author:** Prantika Biswas

## Overview

This project demonstrates the process of building a complete web scraping workflow using Python and Scrapy. The project focuses on extracting structured book data from the Books to Scrape website, including book titles, prices, ratings, availability information, categories, descriptions, product identifiers, and URLs.

The notebook follows a step-by-step learning approach, beginning with basic Scrapy selectors and progressing toward advanced topics such as following links, scraping detail pages, handling pagination, and crawling large datasets across multiple pages.

The primary objective of this project is to understand how modern web scraping frameworks collect, process, and organize data from websites in a scalable and maintainable way.

## Learning Objectives

* Understand the architecture of a Scrapy project
* Work with CSS selectors and response objects
* Extract structured data from web pages
* Follow links between pages
* Scrape detailed information from individual product pages
* Handle pagination across multiple pages
* Export scraped data into JSON format
* Build reusable and maintainable scraping workflows

## Note on Documentation

Throughout the notebook, explanatory text cells have been included alongside the code. These notes were intentionally added to summarize important concepts, explain the purpose of specific code blocks, and provide quick references for future revision. Their purpose is to improve readability, learning, and maintainability of the project for both the author and anyone reviewing or contributing to the repository.


In [ ]:
# Mount and Authorize Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Create Project Folder

import os

project_path = "/content/drive/MyDrive/Project_02_Scrapy_MongoDB"

os.makedirs(project_path, exist_ok=True)

print("Project folder created!")

Project folder created!


In [ ]:
# Move into Project Folder

%cd /content/drive/MyDrive/Project_02_Scrapy_MongoDB

/content/drive/MyDrive/Project_02_Scrapy_MongoDB


Scrapy Projects follow a fixed structure and are built around an ETL Workflow:


*   Extract → Scrapy Spider
*   Transform → Item Pipeline
*   Load → MongoDB






In [ ]:
# Install Scrapy

!pip install scrapy

In [ ]:
# Verify Installation

!scrapy version

Scrapy 2.16.0


In [ ]:
# Create Scrapy Project

!scrapy startproject books

Error: scrapy.cfg already exists in /content/drive/MyDrive/Project_02_Scrapy_MongoDB/books


In [ ]:
# Inspect the Generated Structure

!find books -type f

books/scrapy.cfg
books/books/settings.py
books/books/spiders/__init__.py
books/books/spiders/__pycache__/__init__.cpython-312.pyc
books/books/spiders/__pycache__/bookspider.cpython-312.pyc
books/books/spiders/bookspider.py
books/books/pipelines.py
books/books/middlewares.py
books/books/__init__.py
books/books/__pycache__/__init__.cpython-312.pyc
books/books/__pycache__/settings.cpython-312.pyc
books/books/__pycache__/items.cpython-312.pyc
books/books/items.py
books/books.json
books/detailed_books.json


**What is Scrapy?**

A Python framework for large-scale web scraping.

**What is a Spider?**

A class that knows:



*   where to crawl
*   what to extract
*   which links to follow


**What is ETL?**

Extract → Transform → Load

**What is MongoDB?**

A NoSQL database that stores JSON-like documents instead of SQL tables.

**Inspect the Website You Want to Scrape**

Before writing a scraper, understand:


*   What data we want
*   Where it lives in the HTML
*   How Scrapy can find it

**What Are We Scraping?**

Target Site:
https://books.toscrape.com/

Each Book Card Contains:



*   Image
*   Title
*   Price
*   Rating





In [ ]:
# Cleanup old exports

!rm -f books.json
!rm -f books_v2.json
!rm -f detailed_books.json
!rm -f all_books.json

In [ ]:
# Understand HTML Structure

import requests

url = "https://books.toscrape.com/"

html = requests.get(url).text

print(html[:3000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="static/oscar/favicon.

In [ ]:
# Install BeautifulSoup

!pip install beautifulsoup4

In [ ]:
# Find Book Containers

from bs4 import BeautifulSoup
import requests

html = requests.get("https://books.toscrape.com/").text

soup = BeautifulSoup(html, "html.parser")

books = soup.find_all("article")

print("Books found:", len(books))

Books found: 20


In [ ]:
# Inspect One Book

print(books[0].prettify()[:3000])

# HTML Container: <article class="product_pod">
# Each book lives inside one of these
# CSS Selector: article.product_pod
# Selects every book

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [ ]:
# Extract the Title Manually
# Title Location: <h3> <a title="...">

title = books[0].find("h3").find("a")["title"]

print(title)

A Light in the Attic


In [ ]:
# Extract the Price Manually
# Price Location: <p class="price_color">

price = books[0].find("p", class_="price_color").text

print(price)

Â£51.77


In [ ]:
# Extract Rating Manually
# Rating Location: <p class="star-rating Three">

rating = books[0].find("p")["class"]

print(rating)

['star-rating', 'Three']


Scrapy heavily uses CSS Selectors

Some common selectors:

| CSS Selector        | Meaning                        |
| ------------------- | ------------------------------ |
| article             | all article tags               |
| h3                  | all h3 tags                    |
| a                   | all links                      |
| .price_color        | class named price_color        |
| article.product_pod | article with class product_pod |


In [ ]:
# Verify CSS Selectors Using BeautifulSoup

books = soup.select("article.product_pod")

print(len(books))

20


In [ ]:
# Create a Scrapy Selector

from scrapy import Selector
import requests

html = requests.get("https://books.toscrape.com/").text

selector = Selector(text=html)

print("Selector ready!")

Selector ready!


**What is Selector?**

HTML -->
Scrapy Selector -->
CSS Queries -->
Extract Data

In [ ]:
# Select All Books

books = selector.css("article.product_pod")

print("Books found:", len(books))

Books found: 20


In [ ]:
# Extract the First Title using Scrapy

title = books[0].css("h3 a::attr(title)").get()

print(title)

A Light in the Attic


In [ ]:
# Extract Price using Scrapy

price = books[0].css(".price_color::text").get()

print(price)

Â£51.77


In [ ]:
# Extract Rating Using Scrapy

rating = books[0].css("p.star-rating::attr(class)").get()

print(rating)

star-rating Three


In [ ]:
# Extract All Titles At Once

titles = selector.css(
    "article.product_pod h3 a::attr(title)"
).getall()

print("Number of titles:", len(titles))
print(titles[:5])

Number of titles: 20
['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History of Humankind']


In [ ]:
# Extract Book URLs

urls = selector.css(
    "article.product_pod h3 a::attr(href)"
).getall()

print(urls[:5])

['catalogue/a-light-in-the-attic_1000/index.html', 'catalogue/tipping-the-velvet_999/index.html', 'catalogue/soumission_998/index.html', 'catalogue/sharp-objects_997/index.html', 'catalogue/sapiens-a-brief-history-of-humankind_996/index.html']


**Scrapy Selector:** Selector(text=html)

converts HTML into something Scrapy can query.

**CSS Extraction:** .css(...)

finds elements.

**Attributes:** ::attr(title)

extracts attributes.

**Text:** ::text extracts text.

**Single vs Multiple:**

.get()

.getall()


Website

   ↓

Spider visits page

   ↓

Extracts data

   ↓
   
Returns structured records

In [ ]:
# Move into the project directory

%cd /content/drive/MyDrive/Project_02_Scrapy_MongoDB/books

/content/drive/MyDrive/Project_02_Scrapy_MongoDB/books


In [ ]:
# Check if spider folder present, if not create.

!ls books/spiders

bookspider.py  __init__.py  __pycache__


In [ ]:
# Create the Spider

%%writefile books/spiders/bookspider.py

import scrapy


class BookSpider(scrapy.Spider):

    name = "books"

    start_urls = [
        "https://books.toscrape.com/"
    ]

    def parse(self, response):

        for book in response.css("article.product_pod"):

            yield {
                "title": book.css(
                    "h3 a::attr(title)"
                ).get(),

                "price": book.css(
                    ".price_color::text"
                ).get(),

                "rating": book.css(
                    "p.star-rating::attr(class)"
                ).get()
            }

Overwriting books/spiders/bookspider.py


In [ ]:
# Export Results to JSON

!rm -f books.json
!scrapy crawl books -o books.json

2026-06-15 14:20:18 [scrapy.utils.log] INFO: Scrapy 2.16.0 started (bot: books)
2026-06-15 14:20:18 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.1',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '26.4.0',
 'Python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]',
 'pyOpenSSL': '26.2.0 (OpenSSL 4.0.1 9 Jun 2026)',
 'cryptography': '48.0.1',
 'Platform': 'Linux-6.6.122+-x86_64-with-glibc2.35'}
2026-06-15 14:20:18 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-06-15 14:20:18 [asyncio] DEBUG: Using selector: EpollSelector
2026-06-15 14:20:18 [scrapy.addons] INFO: Enabled addons:
[]
2026-06-15 14:20:18 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-06-15 14:20:18 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-06-15 14:20:18 [scrapy.extensions.telnet] INFO: Telnet Password: 2eac7e14f00b8d9e
2026-06-15 14:20:18 [scrapy.

In [ ]:
# Verify Output File

import json

with open("books.json") as f:
    data = json.load(f)

print("Books scraped:", len(data))
print(data[0])

Books scraped: 20
{'title': 'A Light in the Attic', 'price': '£51.77', 'rating': 'star-rating Three'}


In [ ]:
# Open items.py

!cat books/items.py


import scrapy


class BookItem(scrapy.Item):

    title = scrapy.Field()

    price = scrapy.Field()

    rating = scrapy.Field()

    url = scrapy.Field()

    description = scrapy.Field()

    category = scrapy.Field()

    upc = scrapy.Field()

    availability = scrapy.Field()


In [ ]:
# Replace items.py

%%writefile books/items.py

import scrapy


class BookItem(scrapy.Item):

    title = scrapy.Field()

    price = scrapy.Field()

    rating = scrapy.Field()

    url = scrapy.Field()

Overwriting books/items.py


A Scrapy item is similar to:
```
class Student:
    name
    age
```
but designed specifically for scraped data





In [ ]:
# Update the Spider

%%writefile books/spiders/bookspider.py

import scrapy

from books.items import BookItem


class BookSpider(scrapy.Spider):

    name = "books"

    start_urls = [
        "https://books.toscrape.com/"
    ]

    def parse(self, response):

        for book in response.css("article.product_pod"):

            item = BookItem()

            item["title"] = book.css(
                "h3 a::attr(title)"
            ).get()

            item["price"] = book.css(
                ".price_color::text"
            ).get()

            item["rating"] = book.css(
                "p.star-rating::attr(class)"
            ).get()

            item["url"] = response.urljoin(
                book.css(
                    "h3 a::attr(href)"
                ).get()
            )

            yield item

Overwriting books/spiders/bookspider.py


In [ ]:
!rm -f books_v2.json
!scrapy crawl books -o books_v2.json

2026-06-15 14:20:21 [scrapy.utils.log] INFO: Scrapy 2.16.0 started (bot: books)
2026-06-15 14:20:21 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.1',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '26.4.0',
 'Python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]',
 'pyOpenSSL': '26.2.0 (OpenSSL 4.0.1 9 Jun 2026)',
 'cryptography': '48.0.1',
 'Platform': 'Linux-6.6.122+-x86_64-with-glibc2.35'}
2026-06-15 14:20:21 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-06-15 14:20:21 [asyncio] DEBUG: Using selector: EpollSelector
2026-06-15 14:20:21 [scrapy.addons] INFO: Enabled addons:
[]
2026-06-15 14:20:21 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-06-15 14:20:21 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-06-15 14:20:21 [scrapy.extensions.telnet] INFO: Telnet Password: 69967a4a7f3a86b1
2026-06-15 14:20:21 [scrapy.

In [ ]:
# Inspect Output

import json

with open("books_v2.json") as f:
    data = json.load(f)

print("Books:", len(data))
print(data[0])

Books: 20
{'title': 'A Light in the Attic', 'price': '£51.77', 'rating': 'star-rating Three', 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


**Scrapy Item:**  
class BookItem(scrapy.Item):

Creates a schema for scraped data

**scrapy.Field():**

Defines attributes of the item

**response.urljoin():**

Converts relative URLs
(catalogue/a-light-in-the-attic_1000/index.html) into absolute URLs (https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html).

**yield item:**

Sends a structured object into Scrapy's processing pipeline.

In [ ]:
# Update item model

%%writefile books/items.py

import scrapy


class BookItem(scrapy.Item):

    title = scrapy.Field()

    price = scrapy.Field()

    rating = scrapy.Field()

    url = scrapy.Field()

    description = scrapy.Field()

    category = scrapy.Field()

    upc = scrapy.Field()

    availability = scrapy.Field()

Overwriting books/items.py


In [ ]:
# Replace Spider

%%writefile books/spiders/bookspider.py

import scrapy

from books.items import BookItem


class BookSpider(scrapy.Spider):

    name = "books"

    start_urls = [
        "https://books.toscrape.com/"
    ]

    def parse(self, response):

        for book in response.css("article.product_pod"):

            book_url = response.urljoin(
                book.css(
                    "h3 a::attr(href)"
                ).get()
            )

            yield response.follow(
                book_url,
                callback=self.parse_book
            )

    def parse_book(self, response):

        item = BookItem()

        item["title"] = response.css(
            ".product_main h1::text"
        ).get()

        item["price"] = response.css(
            ".price_color::text"
        ).get()

        item["availability"] = response.css(
            ".availability::text"
        ).getall()

        item["description"] = response.css(
            "#product_description + p::text"
        ).get()

        item["category"] = response.css(
            ".breadcrumb li a::text"
        ).getall()

        item["upc"] = response.css(
            "table tr td::text"
        ).get()

        item["url"] = response.url

        yield item

Overwriting books/spiders/bookspider.py


In [ ]:
# Run spider
!rm -f detailed_books.json
!scrapy crawl books -o detailed_books.json

2026-06-15 14:20:23 [scrapy.utils.log] INFO: Scrapy 2.16.0 started (bot: books)
2026-06-15 14:20:23 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.1',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '26.4.0',
 'Python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]',
 'pyOpenSSL': '26.2.0 (OpenSSL 4.0.1 9 Jun 2026)',
 'cryptography': '48.0.1',
 'Platform': 'Linux-6.6.122+-x86_64-with-glibc2.35'}
2026-06-15 14:20:23 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-06-15 14:20:23 [asyncio] DEBUG: Using selector: EpollSelector
2026-06-15 14:20:23 [scrapy.addons] INFO: Enabled addons:
[]
2026-06-15 14:20:23 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-06-15 14:20:23 [scrapy.utils.log] DEBUG: Using asyncio event loop: asyncio.unix_events._UnixSelectorEventLoop
2026-06-15 14:20:23 [scrapy.extensions.telnet] INFO: Telnet Password: 0c8f8d2054aceb4d
2026-06-15 14:20:23 [scrapy.

In [ ]:
# Check Output

import json

with open("detailed_books.json") as f:
    data = json.load(f)

print("Books:", len(data))

print(data[0])

Books: 20
{'title': "It's Only the Himalayas", 'price': '£45.17', 'availability': ['\n    ', '\n    \n        In stock (19 available)\n    \n', '\n    ', '\n    \n        In stock\n    \n', '\n    ', '\n    \n        In stock\n    \n', '\n    ', '\n    \n        In stock\n    \n', '\n    ', '\n    \n        In stock\n    \n', '\n    ', '\n    \n        In stock\n    \n', '\n    ', '\n    \n        In stock\n    \n'], 'description': '“Wherever you go, whatever you do, just . . . don’t do anything stupid.” —My MotherDuring her yearlong adventure backpacking from South Africa to Singapore, S. Bedford definitely did a few things her mother might classify as "stupid." She swam with great white sharks in South Africa, ran from lions in Zimbabwe, climbed a Himalayan mountain without training in Nepal, and wa “Wherever you go, whatever you do, just . . . don’t do anything stupid.” —My MotherDuring her yearlong adventure backpacking from South Africa to Singapore, S. Bedford definitely did a fe

**response.follow()** means:

Don't extract yet.
Visit this page first.

**callback=self.parse_book** means:

After downloading the book page,
run parse_book()

**Flow:**

Homepage -->
parse() -->
follow URL -->
parse_book() -->
yield item

In [ ]:
# Find the next page selector

from scrapy import Selector
import requests

html = requests.get(
    "https://books.toscrape.com/catalogue/page-1.html"
).text

sel = Selector(text=html)

next_page = sel.css(
    "li.next a::attr(href)"
).get()

print(next_page)

page-2.html


In [ ]:
# Update Spider Logic

%%writefile books/spiders/bookspider.py

import scrapy

from books.items import BookItem


class BookSpider(scrapy.Spider):

    name = "books"

    start_urls = [
        "https://books.toscrape.com/"
    ]

    def parse(self, response):

        # Visit each book page

        for book in response.css("article.product_pod"):

            book_url = response.urljoin(
                book.css(
                    "h3 a::attr(href)"
                ).get()
            )

            yield response.follow(
                book_url,
                callback=self.parse_book
            )

        # Find next page

        next_page = response.css(
            "li.next a::attr(href)"
        ).get()

        if next_page:

            yield response.follow(
                next_page,
                callback=self.parse
            )

    def parse_book(self, response):

        item = BookItem()

        item["title"] = response.css(
            ".product_main h1::text"
        ).get()

        item["price"] = response.css(
            ".price_color::text"
        ).get()

        item["availability"] = (
            "".join(
                response.css(
                    ".availability::text"
                ).getall()
            )
            .strip()
        )

        item["description"] = response.css(
            "#product_description + p::text"
        ).get()

        item["category"] = response.css(
            ".breadcrumb li a::text"
        ).getall()[-1]

        item["upc"] = response.css(
            "table tr td::text"
        ).get()

        item["url"] = response.url

        yield item

Overwriting books/spiders/bookspider.py


In [ ]:
!scrapy crawl books -o all_books.json

Streaming output truncated to the last 5000 lines.
                'down those who already have. The real key to making America a '
                'freer, fairer, more prosperous nation is to protect and '
                'celebrate the pursuit of success—not pull down the high '
                'fliers in the name of equality. ...more',
 'price': '£56.86',
 'title': "Equal Is Unfair: America's Misguided Fight Against Income "
          'Inequality',
 'upc': '3968e3fbf4695d7c',
 'url': 'https://books.toscrape.com/catalogue/equal-is-unfair-americas-misguided-fight-against-income-inequality_617/index.html'}
2026-06-15 14:42:28 [scrapy.core.engine] DEBUG: Crawled (200) <GET https://books.toscrape.com/catalogue/everyday-italian-125-simple-and-delicious-recipes_618/index.html> (referer: https://books.toscrape.com/catalogue/page-20.html)
2026-06-15 14:42:28 [scrapy.core.scraper] DEBUG: Scraped from <200 https://books.toscrape.com/catalogue/everyday-italian-125-simple-and-delicious-recipes_6

In [ ]:
# Verify

import json

with open("all_books.json") as f:
    data = json.load(f)

print("Total Books:", len(data))

Total Books: 1000


In [ ]:
# Check a random record

print(data[500])

{'title': 'Booked', 'price': '£17.49', 'availability': 'In stock (5 available)\n    \n\n    \n    \n        In stock\n    \n\n    \n    \n        In stock\n    \n\n    \n    \n        In stock\n    \n\n    \n    \n        In stock\n    \n\n    \n    \n        In stock\n    \n\n    \n    \n        In stock', 'description': 'Like lightning/you strike/fast and free/legs zoom/down field/eyes fixed/on the checkered ball/on the goal/ten yards to go/can’t nobody stop you/can’t nobody cop you… In this follow-up to the Newbery-winning novel THE CROSSOVER, soccer, family, love, and friendship, take center stage as twelve-year-old Nick learns the power of words as he wrestles with problems at home, sta Like lightning/you strike/fast and free/legs zoom/down field/eyes fixed/on the checkered ball/on the goal/ten yards to go/can’t nobody stop you/can’t nobody cop you… In this follow-up to the Newbery-winning novel THE CROSSOVER, \xa0soccer, family, love, and friendship, take center stage as twelve-y

**Recursive Crawling :**

```
yield response.follow(
    next_page,
    callback=self.parse
)
```
means calling parse() from inside parse() which creates Page 1 --> Page 2 --> Page 3 --> ... --> Page 50 automatically

**Pagination: **

Scraping multiple pages automatically

# Conclusion

In this project, a complete web scraping workflow was implemented using Scrapy. Starting from basic selector extraction, the scraper was gradually enhanced to collect detailed information from individual book pages and navigate through multiple pages using pagination.

The final scraper successfully extracted large-scale structured datasets from the target website and exported the results into JSON format for further analysis and processing.

This project provided practical experience with:

* Scrapy project organization
* Data extraction techniques
* Link traversal and page navigation
* Pagination handling
* Large-scale crawling workflows
* Structured data export

While the original learning path also explored MongoDB integration, database storage has been left as a future enhancement. The current implementation focuses on mastering the scraping pipeline and data collection process before introducing database connectivity.

Future improvements may include:

* MongoDB Atlas integration
* Data validation pipelines
* Duplicate detection mechanisms
* Automated scraping schedules
* Additional export formats and analytics workflows

Overall, the project successfully demonstrates the end-to-end process of collecting, organizing, and exporting web data using Scrapy and serves as a strong foundation for more advanced web scraping and data engineering projects.
